# 第24章 窗口计算与探索性分析

结合滚动窗口、累计指标和描述性统计完成探索性分析。

## 本章定位

本章按“概念 → 示例 → 练习”的顺序组织。代码单元格可以单独运行，也可以从上到下完整运行。

## 学习目标

- 计算滚动与累计指标
- 生成描述性统计
- 检查频数和分位数
- 分析相关关系但避免因果误读


## 核心概念

- 窗口大小必须与业务周期一致。
- 滚动统计前要按时间排序。
- 相关系数只描述线性共同变化，不能证明因果。


## 示例 1：滚动与累计计算

min_periods控制窗口不足时是否返回结果。


In [ ]:
import pandas as pd

sales = pd.DataFrame({
    "date": pd.date_range("2026-01-01", periods=10, freq="D"),
    "amount": [120, 135, 128, 160, 175, 168, 190, 205, 198, 220],
})
sales["rolling_3d"] = sales["amount"].rolling(3, min_periods=1).mean()
sales["cumulative"] = sales["amount"].cumsum()
sales["daily_growth"] = sales["amount"].pct_change()
print(sales.round(3))


## 示例 2：描述性统计与频数

数值概览和类别频数应一起检查。


In [ ]:
orders = pd.DataFrame({
    "region": ["华东", "华南", "华东", "华北", "华东", "华南"],
    "amount": [320, 880, 460, 1250, 720, 540],
    "items": [2, 4, 1, 5, 3, 2],
})
print(orders[["amount", "items"]].describe().round(2))
print(orders["region"].value_counts(normalize=True).round(3))


## 示例 3：相关与分位数

先检查散点和异常，再解释相关系数。


In [ ]:
print("分位数:\n", orders["amount"].quantile([0.25, 0.5, 0.75]))
print("相关矩阵:\n", orders[["amount", "items"]].corr().round(3))
orders["amount_rank"] = orders["amount"].rank(ascending=False, method="dense")
print(orders.sort_values("amount_rank"))


## 初学者学习路线

这章建议按照“先观察、再模仿、后修改、最后独立完成”的顺序学习，不必一次记住所有参数。

1. 先阅读任务说明，明确这段代码要回答什么问题。
2. 运行一个最小例子，先观察输入、输出和数据形状，再回看每一行代码。
3. 只修改一个参数或一条数据，重新运行并比较前后结果。
4. 完成“综合练习”，最后再看本章小结，把能迁移到其他数据的问题写下来。

运行时如果看到 NameError，通常是前置单元格还没有运行；如果输出和预期不同，先检查变量是否被后面的单元格重新赋值。


## 先做一个小检查

进入正式例子前，先用一句话回答：本章的输入是什么，想得到什么结果？

本章主题是“第24章 窗口计算与探索性分析”。请特别留意三件事：输入的类型或形状、处理中间变量的含义、最后输出能否支持一个清楚的结论。


## Pandas 的学习主线

写数据字典 → 读取与检查 → 清洗类型和缺失值 → 选择与筛选 → 新增计算列 → 分组聚合 → 合并或透视 → 导出可复用结果

每一步都说明“一行代表什么”。处理前后记录行数、列数和关键字段；汇总前先确认分组粒度，避免得到数字却无法解释。


## 本模块练习方式

基础：完成一个字段清洗；提高：从明细表生成汇总表；挑战：处理重复、缺失和类型混乱，并写出清洗规则。

完成后请写下：输入是什么、处理做了什么、输出说明了什么、还存在什么限制。


## 示例 4：从明细表生成计算列

这一组例子只处理一个小问题。先运行代码，再逐行对照拆解说明。


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["华东", "华南", "华东"],
    "sales": [120, 150, 180],
    "cost": [80, 100, 130],
})
orders["profit"] = orders["sales"] - orders["cost"]
orders["profit_rate"] = orders["profit"] / orders["sales"]
print(orders.round(3))


### 逐步拆解

先新增一个简单指标，再基于它计算比例；拆成多列可以保留中间结果并方便检查。

建议第一次运行后只改一个输入值，再观察哪一个输出发生变化。


## 示例 5：从明细汇总到业务表

看懂上一个例子后，再观察同一主题在另一种数据或场景中的写法。


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["华东", "华东", "华南", "华南"],
    "channel": ["线上", "线下", "线上", "线下"],
    "sales": [120, 80, 150, 100],
})
summary = orders.groupby(["region", "channel"], as_index=False)["sales"].sum()
print(summary)
print("地区合计：")
print(orders.groupby("region")["sales"].sum())


### 逐步拆解

先明确每一行的粒度，再选择 groupby 的字段；汇总表的每一行代表一个清晰的分组组合。

自我检查：如果把输入数量、类别或参数改成另一组值，代码是否仍然能运行？


## 教学实验：分组汇总与粒度

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["华东", "华东", "华南", "华南"],
    "channel": ["线上", "线下", "线上", "线下"],
    "sales": [120, 80, 150, 100],
})
summary = orders.groupby("region", as_index=False)["sales"].sum()
print(summary)
print("汇总表每一行代表一个地区")


### 第一个结果怎么读

先确认明细表一行代表一笔订单，再确认汇总表一行代表一个地区。`groupby` 的字段决定结果的粒度。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
orders["sales_level"] = orders["sales"].map(
    lambda value: "高" if value >= 120 else "普通"
)
print(orders)
print(orders["sales_level"].value_counts())


### 第二个结果怎么读

第二个实验只增加一个分类列，不改变原始销售额。练习解释：什么时候应该新增列，什么时候应该直接筛选行？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：脏数据转换怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

raw = pd.Series(["12", "unknown", "18", ""])
converted = pd.to_numeric(raw, errors="coerce")
print("转换结果：")
print(converted)
print("无法转换的数量：", converted.isna().sum())
print("后续可以选择删除、填充或回查原始值。")


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

errors="coerce" 会把无法转换的值记录为缺失，适合先完成质量盘点；不要在没有统计数量前直接删除。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 常见误区

- 未按日期排序就计算滚动窗口
- 窗口长度与业务周期不一致
- 把相关系数解释为因果效应


## 综合练习

1. 创建12个月销售序列
2. 计算3个月移动平均和累计销售
3. 找出销售额最高的3个月

请先独立完成，再点击下方“显示答案”查看参考代码。


In [ ]:
import pandas as pd

monthly = pd.DataFrame({
    "month": pd.period_range("2025-01", periods=12, freq="M").astype(str),
    "sales": [120, 128, 135, 142, 150, 146, 160, 172, 180, 188, 205, 218],
})

# TODO: 计算3个月移动平均
monthly["moving_3m"] = monthly["sales"].rolling(3).mean()

# TODO: 计算累计销售
monthly["cumulative"] = monthly["sales"].cumsum()

print(monthly.round(2))
print("Top 3:\n", monthly.nlargest(3, "sales")[["month", "sales"]])


In [ ]:
import pandas as pd

monthly = pd.DataFrame({
    "month": pd.period_range("2025-01", periods=12, freq="M").astype("string"),
    "sales": [120, 128, 135, 142, 150, 146, 160, 172, 180, 188, 205, 218],
})
monthly["moving_3m"] = monthly["sales"].rolling(3, min_periods=1).mean()
monthly["cumulative"] = monthly["sales"].cumsum()
print(monthly.round(2))
print("Top 3:\n", monthly.nlargest(3, "sales")[["month", "sales"]])

# 自检
assert monthly["cumulative"].iloc[-1] == 1944, "检查累计销售：最后一行应该是总和"
assert len(monthly.nlargest(3, "sales")) == 3, "检查Top 3行数"


## 本章小结

结合滚动窗口、累计指标和描述性统计完成探索性分析。

**迁移思考**：

1. 如果需要计算7日移动平均但前6天数据不足，min_periods 应该设置为多少？
2. 为什么相关系数高不代表因果关系？请举一个相关但无因果的例子。


### 你已经掌握

- 计算滚动与累计指标
- 生成描述性统计
- 检查频数和分位数
- 分析相关关系但避免因果误读


### 关键知识速查

| 知识点 | 作用与提醒 | 关键写法 |
| --- | --- | --- |
| 滚动与累计计算 | min_periods控制窗口不足时是否返回结果。 | `pd.DataFrame()`、`pd.date_range()`、`sales.round()`、`.rolling()` |
| 描述性统计与频数 | 数值概览和类别频数应一起检查。 | `pd.DataFrame()`、`.describe()`、`.round()`、`.value_counts()` |
| 相关与分位数 | 先检查散点和异常，再解释相关系数。 | `orders.sort_values()`、`.quantile()`、`.corr()`、`.round()` |


### 需要注意

- 未按日期排序就计算滚动窗口
- 窗口长度与业务周期不一致
- 把相关系数解释为因果效应


### 完成检查

- [ ] 能够计算滚动与累计指标
- [ ] 能够生成描述性统计
- [ ] 能够检查频数和分位数
- [ ] 能够分析相关关系但避免因果误读
